In [1]:
# Import packages
import os
import cv2
import mediapipe as mp
import pandas as pd
from tqdm import tqdm

2026-02-20 00:03:25.766900: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-20 00:03:25.766987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-20 00:03:25.768756: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-20 00:03:25.777869: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Configure directories
UCF_VIDEOS = ""
UCF_SPLITS = "UCF101TrainTestSplits/ucfTrainTestlist"
FRAMES_DIR = "outputs/frames"
KEYPOINTS_DIR = "outputs/keypoints"

# Convert UCF101 classes to our labels
EXERCISE_CLASSES = {
    "PushUps": "pushups", 
    "BodyWeightSquats": "squats",
    "Lunges": "lunges",
    "PullUps": "pullups",
    "JumpingJack": "jumping_jacks"
}

# Configure desired frame rates and image size
TARGET_FPS = 10
IMG_SIZE = (640, 480)

In [3]:
# Filter 5 classes from train/test 01
# We are just using one train/test split for this project

def load_videos():
    rows = []
    for split, filename in [("train", "trainlist01.txt"), ("test", "testlist01.txt")]:
        filepath = os.path.join(UCF_SPLITS, filename)
        with open(filepath) as f:
            for line in f: # e.g. BodyWeightSquats/v_BodyWeightSquats_g08_c01.avi
                rel_path = line.strip().split()[0]
                class_name = rel_path.split("/")[0]
                if class_name not in EXERCISE_CLASSES:
                    continue
                rows.append({
                    "video_path": os.path.join(UCF_VIDEOS, rel_path),
                    "label": EXERCISE_CLASSES[class_name],
                    "split": split,
                    "video_name": os.path.splitext(os.path.basename(rel_path))[0]
                })
    df = pd.DataFrame(rows)
    print(f"Found {len(df)} exercise videos\n{df['label'].value_counts()}\n")
    return df

In [4]:
# Extract frames from the videos
def extract_frames(video_df):
    rows = []
    for _, row in tqdm(video_df.iterrows(), total=len(video_df), desc="Extracting keyframes"):
        save_dir = os.path.join(FRAMES_DIR, row["split"], row["label"], row["video_name"])
        os.makedirs(save_dir, exist_ok=True)

        # Use OpenCV to open the .avi file
        # Checks the video's native FPS and determines how many frames to get based on that
        cap = cv2.VideoCapture(row["video_path"])
        fps = cap.get(cv2.CAP_PROP_FPS) or TARGET_FPS
        frame_interval = max(1, int(round(fps / TARGET_FPS)))

        frame_idx = 0 # Tracks every frame seens
        saved_idx = 0 # Tracks only the frames to save

        while True:
            ret, frame = cap.read()
            if not ret:
                break # Ret = False once the video has ended, so break
            if frame_idx % frame_interval == 0: # Resize and save frames in wanted interval
                frame_path = os.path.join(save_dir, f"frame_{saved_idx:04d}.jpg")
                cv2.imwrite(frame_path, cv2.resize(frame, IMG_SIZE))
                rows.append({**row, "frame_path": frame_path, "frame_index": saved_idx})
                saved_idx += 1
            frame_idx += 1
        cap.release() # Closes the video
    df = pd.DataFrame(rows)
    print(f"Extracted {len(df)} frames total\n")
    return df

In [5]:
# Name keypoints according to MediaPipe Pose standards
KEYPOINT_NAMES = [
    "nose", "left_eye_inner", "left_eye", "left_eye_outer",
    "right_eye_inner", "right_eye", "right_eye_outer",
    "left_ear", "right_ear", "mouth_left", "mouth_right",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_pinky", "right_pinky",
    "left_index", "right_index", "left_thumb", "right_thumb",
    "left_hip", "right_hip", "left_knee", "right_knee",
    "left_ankle", "right_ankle", "left_heel", "right_heel",
    "left_foot_index", "right_foot_index",
]

# Extract keypoints 
def extract_keypoints(frame_df):
    os.makedirs(KEYPOINTS_DIR, exist_ok=True)
    mp_pose = mp.solutions.pose # Load MediaPipe Pose module
    rows = []
    with mp_pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.5) as pose:
        # Loop through each frame in the dataframe
        for _, row in tqdm(frame_df.iterrows(), total=len(frame_df), desc="Pose estimation"):
            frame = cv2.imread(row["frame_path"])
            if frame is None:
                continue
            
            # Run MediaPipe on the frame after converting to RGB
            results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if results is None:
                continue

            keypoints = {}
            for i, name in enumerate(KEYPOINT_NAMES):
                kp = results.pose_landmarks.landmark[i]
                # x and y-> position on screen
                keypoints[f"{name}_x"]   = round(kp.x, 6)
                keypoints[f"{name}_y"]   = round(kp.y, 6)

                rows.append({
                    "frame_path": row["frame_path"],
                    "video_name": row["video_name"],
                    "label": row["label"],
                    "split": row["split"],
                    "frame_index": row["frame_index"],
                    **keypoints # Append entire keyframes dictionary at once
                })

    df = pd.DataFrame(rows)
    # Output two CSVs that the model will train on (train and test keypoints)
    for split, group in df.groupby("split"):
        out_path = os.path.join(KEYPOINTS_DIR, f"{split}_keypoints.csv")
        group.to_csv(out_path, index=False)
        print(f"Saved {len(group)} rows → {out_path}")
        print(group["label"].value_counts(), "\n")
    return df

In [8]:
video_df = load_videos()
video_df.head()

Found 564 exercise videos
label
lunges           127
jumping_jacks    123
squats           112
pushups          102
pullups          100
Name: count, dtype: int64



,video_path,label,split,video_name
0,BodyWeightSquats/v_BodyWeightSquats_g08_c01.avi,squats,train,v_BodyWeightSquats_g08_c01
1,BodyWeightSquats/v_BodyWeightSquats_g08_c02.avi,squats,train,v_BodyWeightSquats_g08_c02
2,BodyWeightSquats/v_BodyWeightSquats_g08_c03.avi,squats,train,v_BodyWeightSquats_g08_c03
3,BodyWeightSquats/v_BodyWeightSquats_g08_c04.avi,squats,train,v_BodyWeightSquats_g08_c04
4,BodyWeightSquats/v_BodyWeightSquats_g09_c01.avi,squats,train,v_BodyWeightSquats_g09_c01


In [7]:
#next steps: labeling good vs bad form 

In [9]:
def compute_angle(a, b, c):
    """Angle at point b, formed by a-b-c"""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

def add_angles(df):
    def angle(r, a, b, c):
        """Helper to pull x/y from row and compute angle at b"""
        return compute_angle(
            (r[f"{a}_x"], r[f"{a}_y"]),
            (r[f"{b}_x"], r[f"{b}_y"]),
            (r[f"{c}_x"], r[f"{c}_y"])
        )

    # --- KNEE ANGLES (squats, lunges) ---
    for side in ["left", "right"]:
        df[f"{side}_knee_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_hip", f"{s}_knee", f"{s}_ankle"), axis=1)

    # --- ELBOW ANGLES (push-ups, pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_elbow_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_elbow", f"{s}_wrist"), axis=1)

    # --- HIP ANGLES (squats, lunges, pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_hip_angle"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_hip", f"{s}_knee"), axis=1)

    # --- BODY LINE / HIP SAG (push-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_body_line"] = df.apply(
            lambda r, s=side: angle(r, f"{s}_shoulder", f"{s}_hip", f"{s}_ankle"), axis=1)

    # --- TORSO UPRIGHTNESS (lunges, squats) ---
    # Angle between vertical and the shoulder→hip vector
    df["torso_angle"] = df.apply(lambda r: compute_angle(
        (r["left_shoulder_x"], r["left_shoulder_y"] - 0.1),  # point above shoulder
        (r["left_shoulder_x"], r["left_shoulder_y"]),
        (r["left_hip_x"],      r["left_hip_y"])
    ), axis=1)

    # --- KNEE VALGUS (squats, lunges) ---
    # How far knee is inside/outside the ankle — not an angle, just a diff
    for side in ["left", "right"]:
        df[f"{side}_knee_valgus"] = df[f"{side}_knee_x"] - df[f"{side}_ankle_x"]

    # --- HEAD DROP (push-ups) ---
    df["head_drop"] = df["nose_y"] - df["left_shoulder_y"]

    # --- SHOULDER SHRUG (pull-ups) ---
    for side in ["left", "right"]:
        df[f"{side}_shoulder_shrug"] = df[f"{side}_ear_y"] - df[f"{side}_shoulder_y"]

    # --- SYMMETRY (jumping jacks) ---
    df["arm_symmetry"] = abs(df["left_elbow_angle"] - df["right_elbow_angle"])
    df["leg_symmetry"] = abs(df["left_knee_angle"]  - df["right_knee_angle"])
    df["hip_symmetry"] = abs(df["left_hip_angle"]   - df["right_hip_angle"])

    return df